# SVM

In [21]:
import pandas as pd
import pyarrow.parquet as pq
import gc
from pathlib import Path
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, train_test_split

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [32]:
def calcular_acertos_por_area(df):
    cols_questao = [c for c in df.columns if c.startswith('questao_')]

    for sigla in ['LC', 'MT', 'CH', 'CN']:
        cols_area = [c for c in cols_questao if c.endswith(f'_{sigla}')]
        df[f'ACERTOS_{sigla}'] = df[cols_area].sum(axis=1)

    return df

In [33]:
def amostra_parquet(filepath, n=10000, seed=42):
    pf = pq.ParquetFile(filepath)
    batch = next(pf.iter_batches(batch_size=n * 5))
    return batch.to_pandas().sample(n=n, random_state=seed)

def amostra_pasta(datapath, n=10000, seed=42):
    arquivos = Path(datapath).glob('*.parquet')
    df_reduzido = pd.concat([
        amostra_parquet(arq, n=n, seed=seed)
        for arq in arquivos
    ], ignore_index=True)
    return df_reduzido

In [12]:
DATA_DIR = '/content/drive/MyDrive/ENEM/dados acertos'

In [13]:
#df = pd.read_parquet("/content/drive/MyDrive/ENEM/dados acertos", engine='pyarrow', dtype_backend='pyarrow')

df = amostra_pasta(DATA_DIR, n=5000)

In [14]:
df.shape

(25000, 204)

In [15]:
df = calcular_acertos_por_area(df)

In [16]:
df.shape

(25000, 208)

In [18]:
df['MEDIA_GERAL'] = (
    df['ACERTOS_LC'] +
    df['ACERTOS_MT'] +
    df['ACERTOS_CH'] +
    df['ACERTOS_CN']
) / 4

df['CLASSE'] = df.groupby('NU_ANO')['MEDIA_GERAL'].transform(
    lambda x: pd.qcut(x, q=3, labels=[0, 1, 2])
).astype('Int64')

df['CLASSE'] = df['CLASSE'].astype(int)

In [19]:
features = (
        [f'questao_{i}_LC' for i in range(1, 46)] +
        [f'questao_{i}_CH' for i in range(1, 46)] +
        [f'questao_{i}_CN' for i in range(1, 46)] +
        [f'questao_{i}_MT' for i in range(1, 46)]
    )

X = df[features]
y = df['CLASSE']


In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [27]:
X_train.shape

(16000, 180)

In [28]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', random_state=42))
])

params = {
    'svm__C': [0.1, 1, 10],
    'svm__gamma': ['scale', 'auto']
}

grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)


Fitting 5 folds for each of 6 candidates, totalling 30 fits
Melhores parâmetros: {'svm__C': 10, 'svm__gamma': 'scale'}
              precision    recall  f1-score   support

           0       0.95      0.94      0.95      1677
           1       0.90      0.92      0.91      1687
           2       0.96      0.95      0.96      1636

    accuracy                           0.94      5000
   macro avg       0.94      0.94      0.94      5000
weighted avg       0.94      0.94      0.94      5000



In [30]:
print("Melhores parâmetros:", grid.best_params_)
print(classification_report(y_val, grid.predict(X_val)))

Melhores parâmetros: {'svm__C': 10, 'svm__gamma': 'scale'}
              precision    recall  f1-score   support

           0       0.95      0.95      0.95      1394
           1       0.89      0.91      0.90      1265
           2       0.97      0.95      0.96      1341

    accuracy                           0.94      4000
   macro avg       0.94      0.94      0.94      4000
weighted avg       0.94      0.94      0.94      4000



In [34]:
df = amostra_pasta(DATA_DIR, n=10000)

In [36]:
df = calcular_acertos_por_area(df)

In [37]:
df['MEDIA_GERAL'] = (
    df['ACERTOS_LC'] +
    df['ACERTOS_MT'] +
    df['ACERTOS_CH'] +
    df['ACERTOS_CN']
) / 4

df['CLASSE'] = df.groupby('NU_ANO')['MEDIA_GERAL'].transform(
    lambda x: pd.qcut(x, q=4, labels=[0, 1, 2, 3])
).astype('Int64')

df['CLASSE'] = df['CLASSE'].astype(int)

In [39]:
features = (
        [f'questao_{i}_LC' for i in range(1, 46)] +
        [f'questao_{i}_CH' for i in range(1, 46)] +
        [f'questao_{i}_CN' for i in range(1, 46)] +
        [f'questao_{i}_MT' for i in range(1, 46)]
    )

X = df[features]
y = df['CLASSE']

In [40]:
X.shape

(50000, 180)

In [41]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [43]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', random_state=42))
])

param_grid = {
    'svm__C': [0.1, 1, 5, 10, 50],
    'svm__gamma': ['scale', 'auto']
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('svm', SVC(random_state=42))]),
             n_jobs=-1,
             param_grid={'svm__C': [0.1, 1, 5, 10, 50],
                         'svm__gamma': ['scale', 'auto']},
             scoring='accuracy', verbose=1)

In [44]:
print("Melhores parâmetros:", grid.best_params_)
print(classification_report(y_val, grid.predict(X_val)))

Melhores parâmetros: {'svm__C': 5, 'svm__gamma': 'scale'}
              precision    recall  f1-score   support

           0       0.95      0.93      0.94      2105
           1       0.87      0.89      0.88      2008
           2       0.89      0.90      0.90      1857
           3       0.97      0.95      0.96      2030

    accuracy                           0.92      8000
   macro avg       0.92      0.92      0.92      8000
weighted avg       0.92      0.92      0.92      8000

